# 并发与并行
并发是轮流处理， 并行是同时处理


# 使用多线程

## 创建线程


# 线程同步： 锁， condvar 和信号量
## 共享内存和消息传递的区别
* 共享内存相对消息传递节省了多次内存拷贝的成本
* 共享内存的实现简洁的多
* 共享内存的锁竞争更多

## 互斥锁Mutex
Mutex让多个线程的并发访问变成了排队访问
### 单线程中使用Mutex
```rust
use std::sync::Mutex;
fn main () {
    let m = Mutex::new(5);
    {
        let mut num = m.lock().unwarp();
        *m = 6;
    }
    println!("m = {:?}", m);
}
```
m.lock()会阻塞当前线程直到获取锁。
**m.lock()方法肯呢个报错，例如正在持有锁的线程panic了，在这种情况下，其他线程不可能再获得锁**

m.lock()返回的是一个智能指针MutexGuard
* 它实现了Deref特征， 自动介意用获取到锁内部的数据
* 实现了drop特征

**mutex锁的粒度是变量， 上述num = m.lock().unwarp(). 表明锁被绑定到了num上，在释放锁之前，只能通过num去访问锁里的数据**

### 多线程中的Mutex
使用Arc智能指针包裹住mutex， 这样就能在多线程之间共享了

## 死锁


# 基于send和Sync的线程安全
Rc, RefCell和裸指针不可以在多线程间使用

## Rc和Arc源码对比
```rust
//Rc源码片段
impl<T: ?Sized> !marker::Send for Rc<T> {}
impl<T: ?Sized> !marker::Sync for Rc<T> {}

//Arc源码片段
unsafe impl<T: ?Sized + Sync + Send> Send for Arc<T> {}
unsafe impl<T: ?Sized + Sync + Send> Sync for Arc<T> {}
```
## Send 和Sync
Send和Sync是Rust安全并发的重中之重，实际上它们只是标记特征(marker trait， 该特征未定义任何行为)
* 实现Send的类型可以在线程之间安全传递所有权
* 实现Sync的类型可以在线程间安全共享(通过引用)

**一个类型要在线程安全共享的前提是 指向它的引用必须能在线程间传递，如果引用不能被传递，就无法在多个线程间使用引用去访问同一个数据**

示例：
```rust
unsafe impl<T: ?Sized+Send+Sync> Sync for RwLock<T>{}
```
我们知道读写锁，可以在线程间共享， 且读写锁内容可以高并发读，因此其中的T也必定是实现了Sync

```rust
unsafe impl<T :?Sized +Send> Sync for Mutex<T>{}
```
Mutex可以在多线程间访问， 但其内容却必须互斥访问， 因此Mutex本身实现了Sync， 但T却没有实现Sync

## 实现Send和Sync类型
